# 05 — Predictive Modeling: Churn, CLV, and Expected Profit

**Owner:** M5 — Thư  
**Purpose:** run the professional, script-backed modeling pipeline.

This notebook is intentionally thin. The reusable modeling logic lives in `scripts/modeling.py`, `scripts/evaluation.py`, and `scripts/utils.py`. Keeping the logic in scripts makes the repo easier to maintain, while this notebook remains the reviewable execution record for the group.

## What this notebook does

1. Validates the current repo structure and core input files.
2. Runs the M5 end-to-end pipeline from `config/paths.yaml`.
3. Produces churn model metrics, CLV metrics, customer ranking, and A/B-test handoff files.

The code follows the repo structure planned by the team and does not modify M1–M4 notebooks.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'Data').exists() and (PROJECT_ROOT.parent / 'Data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('Config:', PROJECT_ROOT / 'config' / 'paths.yaml')

## 1. Validate core inputs

In [ ]:
from scripts.data_processing import validate_core_inputs

input_checks = validate_core_inputs(PROJECT_ROOT / 'config' / 'paths.yaml')
input_checks

## 2. Run M5 pipeline

In [ ]:
from scripts.modeling import run_m5_pipeline

summary = run_m5_pipeline(PROJECT_ROOT / 'config' / 'paths.yaml')
summary

## 3. Review generated outputs

In [ ]:
import pandas as pd

model_dir = PROJECT_ROOT / 'models'
metrics = pd.read_csv(model_dir / 'model_metrics.csv')
clv_metrics = pd.read_csv(model_dir / 'clv_model_metrics.csv')
scenario_summary = pd.read_csv(model_dir / 'scenario_profit_summary.csv')

display(metrics[['model', 'val_PR_AUC', 'val_F2_score', 'test_PR_AUC', 'test_F2_score', 'test_precision', 'test_recall']])
display(clv_metrics)
display(scenario_summary)

## 4. Handoff files for M6

Use `models/high_risk_customers_for_ab_test.csv` as the experiment population candidate file. Use `models/profitable_treatment_candidates_base.csv` as the profitability sanity-check file. High churn risk is not the same thing as positive expected treatment profit.